# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example of loading and exploring the FAIR² dataset (mass spectrometry analysis of EVs from human mast cells) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata (access as attributes, not subscript)
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All entities are referenced by their `@id`. We'll print the record set IDs and, for each, their field and column IDs.

In [ ]:
# Access record sets and list their IDs and their fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record set name: {getattr(rs, 'name', None)} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    -> {getattr(field, 'name', None)} (@id: {field.id}) (dataType: {field.data_type})")
            if hasattr(field, 'column') and field.column:
                print(f"       Column @id: {field.column.id}")
    print("")

## 3. Data Extraction
Load data from one or all record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Create a dictionary of DataFrames indexed by record set @id
dataframes = {}

# Iterate over record sets, extracting all records as DataFrame
for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only create DataFrame if there is data
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing, including filtering, normalization, and grouping based on field `@id`s. 
In this example, we'll select a numeric field from one of the main record sets for demonstration.

In [ ]:
# --- Customize the record set and field for EDA (use IDs printed above) ---
# Choose the main record set (edit the @id as appropriate for your exploration)
main_recordset_id = record_set_ids[0] if record_set_ids else None

# Display columns for convenience
if main_recordset_id:
    df = dataframes[main_recordset_id]
    print(f"Columns for {main_recordset_id}: {df.columns.tolist()}")

    # Attempt to auto-detect a numeric field for demonstration
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}\n")
        # EDA: Filtering examples
        threshold = df[numeric_field_id].quantile(0.9)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} above 90th percentile (>{threshold}):")
        display(filtered_df.head())

        # Normalize (Z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by another column (non-numeric)
        candidate_group_field = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                candidate_group_field = col
                break
        if candidate_group_field:
            grouped_df = filtered_df.groupby(candidate_group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {candidate_group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No usable record set found.")

## 5. Visualization
Visualize numeric distributions and relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_recordset_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a candidate group field is available, plot mean by group
    if 'candidate_group_field' in locals() and candidate_group_field:
        plt.figure(figsize=(10,5))
        order = df[candidate_group_field].value_counts().index[:10]
        group_means = df.groupby(candidate_group_field)[numeric_field_id].mean().loc[order]
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {candidate_group_field} (top 10)")
        plt.xlabel(candidate_group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We successfully loaded the FAIR² mass spectrometry dataset using `mlcroissant`, examined its structure, and performed basic filtering, normalization, grouping, and visualization using field and record set `@id`s. For further analysis, consult the schema documentation and adapt field selections for your scientific use-case.